In [2]:
import torch
from torch import Tensor
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pytorch_lightning as pl
from torch.distributions import Bernoulli
import xarray as xr
import xrft
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal
import pandas as pd
import functools as ft
from collections import namedtuple
from IPython.display import Markdown, display
from omegaconf import OmegaConf
import yaml
import inspect
import hydra
import os
from matplotlib.gridspec import GridSpec
from matplotlib.animation import FuncAnimation, PillowWriter
import matplotlib.colors as colors
from matplotlib.colors import LogNorm 
from matplotlib.animation import FuncAnimation
import matplotlib.pyplot as plt
from PIL import Image
import kornia.filters as kfilts
from torch.amp import autocast, GradScaler
from scipy.ndimage import gaussian_filter
from scipy.ndimage import binary_dilation
from skimage.morphology import disk, dilation
from sklearn.linear_model import LinearRegression
import random
import seaborn as sns
from scipy.stats import ks_2samp, wasserstein_distance, energy_distance, linregress, cramervonmises_2samp, anderson_ksamp, rankdata
import pandas as pd
from scipy.ndimage import sobel
from typing import Optional, Tuple, Union

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
#device = "cpu"

In [3]:
def plot_tensor(tensor, title = 'Logits Plot'):
    pos_tensor_numpy = tensor.numpy()
    plt.imshow(pos_tensor_numpy, cmap='viridis', interpolation='nearest')
    plt.colorbar()
    plt.title(title)
    plt.show()

def rmse_based_scores_from_ds(ds, ref_variable='tgt', study_variable='out'):
    #mask = ~np.isnan(ds['input'])
    try:
        return rmse_based_scores(ds[ref_variable], ds[study_variable])[2:]
    except:
        return [np.nan, np.nan]

def psd_based_scores_from_ds(ds, ref_variable='tgt', study_variable='out'):
    print(ds)
    try:
        return psd_based_scores(ds[ref_variable], ds[study_variable])[1:]
    except:
        return [np.nan, np.nan]

def rmse_based_scores_lead(da_rec, da_ref):
    rmse_t = (
        1.0
        - (((da_rec - da_ref) ** 2).mean(dim=("lon", "lat"))) ** 0.5
        / (((da_ref) ** 2).mean(dim=("lon", "lat"))) ** 0.5
    )
    rmse_xy = (((da_rec - da_ref) ** 2).mean(dim=("time"))) ** 0.5
    rmse_t = rmse_t.rename("rmse_t")
    rmse_xy = rmse_xy.rename("rmse_xy")
    reconstruction_error_stability_metric = rmse_t.std().values
    leaderboard_rmse = (
        1.0 - (((da_rec - da_ref) ** 2).mean()) ** 0.5 / (((da_ref) ** 2).mean()) ** 0.5
    )
    return (
        np.round(leaderboard_rmse.values, 5).item(),
        np.round(reconstruction_error_stability_metric, 5).item(),
    )

def rmse_based_scores_day(da_rec, da_ref):
    # Calculate leaderboard RMSE
    leaderboard_rmse = (
        1.0 - (((da_rec - da_ref) ** 2).mean()) ** 0.5 / (((da_ref) ** 2).mean()) ** 0.5
    )
    return np.round(leaderboard_rmse.values, 5).item()

def rmse_based_scores(da_rec, da_ref):
    rmse_t = (
        1.0
        - (((da_rec - da_ref) ** 2).mean(dim=("lon", "lat"))) ** 0.5
        / (((da_ref) ** 2).mean(dim=("lon", "lat"))) ** 0.5
    )
    rmse_xy = (((da_rec - da_ref) ** 2).mean(dim=("time"))) ** 0.5
    rmse_t = rmse_t.rename("rmse_t")
    rmse_xy = rmse_xy.rename("rmse_xy")
    reconstruction_error_stability_metric = rmse_t.std().values
    leaderboard_rmse = (
        1.0 - (((da_rec - da_ref) ** 2).mean()) ** 0.5 / (((da_ref) ** 2).mean()) ** 0.5
    )
    return (
        rmse_t,
        rmse_xy,
        np.round(leaderboard_rmse.values, 5).item(),
        np.round(reconstruction_error_stability_metric, 5).item(),
    )

def psd_based_scores(da_rec, da_ref):
    print('hello')
    err = da_rec - da_ref
    err["time"] = (err.time - err.time[0]) / np.timedelta64(1, "D")
    signal = da_ref
    signal["time"] = (signal.time - signal.time[0]) / np.timedelta64(1, "D")
    psd_err = xrft.power_spectrum(
        err, dim=["time", "lon"], detrend="constant", window="hann"
    ).compute()
    psd_signal = xrft.power_spectrum(
        signal, dim=["time", "lon"], detrend="constant", window="hann"
    ).compute()

    mean_psd_signal = psd_signal.mean(dim="lat").where(
        (psd_signal.freq_lon > 0.0) & (psd_signal.freq_time > 0), drop=True
    )
    
    mean_psd_err = psd_err.mean(dim="lat").where(
        (psd_err.freq_lon > 0.0) & (psd_err.freq_time > 0), drop=True
    )
    print(mean_psd_err)
    psd_based_score = 1.0 - mean_psd_err / mean_psd_signal
    level = [0.5]
    cs = plt.contour(
        1.0 / psd_based_score.freq_lon.values,
        1.0 / psd_based_score.freq_time.values,
        psd_based_score,
        level,
    )
    x05, y05 = cs.collections[0].get_paths()[0].vertices.T
    plt.close()

    shortest_spatial_wavelength_resolved = np.min(x05)
    shortest_temporal_wavelength_resolved = np.min(y05)
    psd_da = 1.0 - mean_psd_err / mean_psd_signal
    psd_da.name = "psd_score"
    return (
        psd_da.to_dataset(),
        np.round(shortest_spatial_wavelength_resolved, 3).item(),
        np.round(shortest_temporal_wavelength_resolved, 3).item(),
    )

def weighted_mse(err, weight = None):
    if weight is None:
        err_w = err
        non_zeros = (torch.ones_like(err) == 0.0)
    else:
        err_w = err * weight[None, ...]
        non_zeros = (torch.ones_like(err) * weight[None, ...]) == 0.0
    err_num = err.isfinite() & ~non_zeros
    if err_num.sum() == 0:
        return torch.scalar_tensor(1000.0, device=err_num.device).requires_grad_()
    loss = F.mse_loss(err_w[err_num], torch.zeros_like(err_w[err_num]))
    return loss

def pprint_cfg(cfg):
    display(Markdown("""```yaml\n\n""" +yaml.dump(OmegaConf.to_container(cfg), default_flow_style=None, indent=2)+"""\n\n```"""))


def update_dz_fig(frame, da, da_4dVN):
    plt.clf()  # Clear the previous frame
    global_min = np.min([da[frame, :, :], da_4dVN[frame, :, :]])
    global_max = np.max([da[frame, :, :], da_4dVN[frame, :, :]])
    
    mu_4dVN = rmse_based_scores_day(da[frame, :, :], da_4dVN[frame, :, :])

    corr_da_4dVN = xr.corr(da[frame, :, :], da_4dVN[frame, :, :])

    # Ground truth
    ax0 = fig.add_subplot(gs[0])
    im0 = ax0.imshow(da[frame, :, :] + 1, extent=[-60, -54, 32, 38], cmap='viridis', vmin=global_min, vmax=global_max)
    ax0.text(0.5, 1.05, 'Ground truth', ha='center', va='bottom', transform=ax0.transAxes, fontsize=14)

    title_4dVarNet = '4dVarNet'
    subtitle_4dVarNet = f'Correlation: {corr_da_4dVN:.2f}\nRMSE Score: {mu_4dVN:.2f}'

    # 4dVarNet
    ax2 = fig.add_subplot(gs[1])
    im2 = ax2.imshow(da_4dVN[frame, :, :] + 1, extent=[-60, -54, 32, 38], cmap='viridis', vmin=global_min, vmax=global_max)
    ax2.text(0.5, 1.1, title_4dVarNet, ha='center', va='bottom', transform=ax2.transAxes, fontsize=14)
    ax2.text(0.5, 1.01, subtitle_4dVarNet, ha='center', va='bottom', transform=ax2.transAxes, fontsize=10, linespacing=1.5)

    # Colorbar
    ax3 = fig.add_subplot(gs[2])
    plt.colorbar(im2, cax=ax3, label='ECS')

    ax3_position = ax3.get_position()
    new_position = [ax3_position.x0 - 0.03, ax3_position.y0, ax3_position.width, ax3_position.height]
    ax3.set_position(new_position)

In [4]:
def _rbf_cov(
    x: Tensor,
    y: Tensor,
    length_scale: Union[float, Tensor],
    sigma_f: Union[float, Tensor],
) -> Tensor:
    """Radial‑basis‑function (squared‑exponential) covariance.

    Parameters
    ----------
    x, y : (N, 2) and (M, 2) tensors of *physical* coordinates (same units)
    length_scale : scalar (>0) – decorrelation length.
    sigma_f : scalar – marginal standard deviation (signal).
    """
    # Ensure tensors and broadcastable shapes
    if not torch.is_tensor(length_scale):
        length_scale = torch.as_tensor(length_scale, dtype=x.dtype, device=x.device)
    if not torch.is_tensor(sigma_f):
        sigma_f = torch.as_tensor(sigma_f, dtype=x.dtype, device=x.device)

    sqdist = ((x[:, None, :] - y[None, :, :]) ** 2).sum(-1)  # (N, M)
    return sigma_f ** 2 * torch.exp(-0.5 * sqdist / (length_scale ** 2))


@torch.no_grad()
def optimal_interpolation(
    data: Tensor,
    *,
    length_scale: float | Tensor = 15.0,
    sigma_f: float | Tensor = 1.0,
    sigma_n: float | Tensor = 0.1,
    coords: Optional[Tensor] = None,
    return_std: bool = False,
    jitter: float = 1e-6,
) -> Tuple[Tensor, Optional[Tensor]]:
    """Optimal interpolation (kriging) on a 2‑D field with missing values.

    Parameters
    ----------
    data : (H, W) tensor
        Field with NaNs at unobserved locations.
    length_scale, sigma_f, sigma_n : float or tensor
        Hyper‑parameters of the RBF kernel and white‑noise error.
    coords : (H, W, 2) tensor or *None*
        Physical coordinates (e.g. lon/lat).  If *None*, pixel indices are used.
    return_std : bool
        If *True*, also return the posterior standard deviation at analysed pixels.
    jitter : float (default 1e‑6)
        Added to the diagonal of *K* for numerical stability.

    Returns
    -------
    analysed : (H, W) tensor      – field with NaNs filled.
    std      : (H, W) tensor or *None* – posterior 1‑σ error (if *return_std*).
    """
    device, dtype = data.device, data.dtype
    H, W = data.shape

    # ------------------------------------------------------------------
    # Coordinates
    if coords is None:
        ii, jj = torch.meshgrid(
            torch.arange(H, device=device, dtype=dtype),
            torch.arange(W, device=device, dtype=dtype),
            indexing="ij",
        )
        coords = torch.stack((jj, ii), dim=-1)  # x = col(j), y = row(i) → shape (H, W, 2)
    else:
        if coords.shape != (H, W, 2):
            raise ValueError("coords must have shape (H, W, 2)")
        coords = coords.to(device=device, dtype=dtype)

    observed_mask = ~torch.isnan(data)
    missing_mask = ~observed_mask

    if observed_mask.sum() == 0:
        raise ValueError("No observed points in `data`. Cannot perform OI.")
    if missing_mask.sum() == 0:  # nothing to fill
        return data.clone(), (None if not return_std else torch.zeros_like(data))

    obs_coords = coords[observed_mask].reshape(-1, 2)
    mis_coords = coords[missing_mask].reshape(-1, 2)
    obs_vals = data[observed_mask].to(dtype)

    # ------------------------------------------------------------------
    # Build covariance matrices
    K_oo = _rbf_cov(obs_coords, obs_coords, length_scale, sigma_f)
    if not torch.is_tensor(sigma_n):
        sigma_n = torch.as_tensor(sigma_n, dtype=dtype, device=device)
    K_oo.diagonal().add_(sigma_n ** 2 + jitter)

    K_mo = _rbf_cov(mis_coords, obs_coords, length_scale, sigma_f)

    # ------------------------------------------------------------------
    # Solve K_oo * alpha = obs_vals (via Cholesky) ----------------------
    L = torch.linalg.cholesky(K_oo)
    alpha = torch.cholesky_solve(obs_vals.unsqueeze(-1), L).squeeze(-1)

    # ------------------------------------------------------------------
    # Posterior mean at missing
    mean_mis = K_mo @ alpha

    analysed = data.clone()
    analysed[missing_mask] = mean_mis

    if not return_std:
        return analysed, None

    # ------------------------------------------------------------------
    # Posterior variance diag = K_mm - K_mo @ K_oo^{-1} @ K_om
    # Compute v = L^{-1} K_om^T  (K_om = K_mo.T)
    v = torch.linalg.solve_triangular(L, K_mo.T, upper=False)
    var_mis = _rbf_cov(mis_coords, mis_coords, length_scale, sigma_f).diagonal() - (v ** 2).sum(0)
    std = torch.zeros_like(data)
    std[missing_mask] = var_mis.clamp_min(0.0).sqrt()

    return analysed, std

In [5]:
tgt_ds = xr.open_dataset('/Odyssey/public/natl60/ssh/NATL60-CJM165-ssh-2012-2013-1_20.nc')
loss_ds = xr.open_dataset('/Odyssey/private/ochapron/ConcreteAE/GS_model/outputs/results_natl60.nc')
logits_ds = xr.open_dataset('/Odyssey/private/ochapron/ConcreteAE/GS_model/outputs/logits_natl60.nc')
prob_ds = xr.open_dataset('/Odyssey/private/ochapron/ConcreteAE/GS_model/outputs/obs_prob_natl60.nc')

loss_ds_2 = xr.open_dataset('/Odyssey/private/ochapron/ConcreteAE/GS_model/outputs_v3/results_natl60_sigma_2.nc')
logits_ds_2 = xr.open_dataset('/Odyssey/private/ochapron/ConcreteAE/GS_model/outputs_v3/logits_natl60_sigma_2.nc')
prob_ds_2 = xr.open_dataset('/Odyssey/private/ochapron/ConcreteAE/GS_model/outputs_v3/obs_prob_natl60_sigma_2.nc')

loss_ds_5 = xr.open_dataset('/Odyssey/private/ochapron/ConcreteAE/GS_model/outputs_v3/results_natl60_sigma_5.nc')
logits_ds_5 = xr.open_dataset('/Odyssey/private/ochapron/ConcreteAE/GS_model/outputs_v3/logits_natl60_sigma_5.nc')
prob_ds_5 = xr.open_dataset('/Odyssey/private/ochapron/ConcreteAE/GS_model/outputs_v3/obs_prob_natl60_sigma_5.nc')

loss_ds_10 = xr.open_dataset('/Odyssey/private/ochapron/ConcreteAE/GS_model/outputs_v3/results_natl60_sigma_10.nc')
logits_ds_10 = xr.open_dataset('/Odyssey/private/ochapron/ConcreteAE/GS_model/outputs_v3/logits_natl60_sigma_10.nc')
prob_ds_10 = xr.open_dataset('/Odyssey/private/ochapron/ConcreteAE/GS_model/outputs_v3/obs_prob_natl60_sigma_10.nc')

loss_ds_15 = xr.open_dataset('/Odyssey/private/ochapron/ConcreteAE/GS_model/outputs_v3/results_natl60_sigma_15.nc')
logits_ds_15 = xr.open_dataset('/Odyssey/private/ochapron/ConcreteAE/GS_model/outputs_v3/logits_natl60_sigma_15.nc')
prob_ds_15 = xr.open_dataset('/Odyssey/private/ochapron/ConcreteAE/GS_model/outputs_v3/obs_prob_natl60_sigma_15.nc')

loss_ds_20 = xr.open_dataset('/Odyssey/private/ochapron/ConcreteAE/GS_model/outputs_v3/results_natl60_sigma_20.nc')
logits_ds_20 = xr.open_dataset('/Odyssey/private/ochapron/ConcreteAE/GS_model/outputs_v3/logits_natl60_sigma_20.nc')
prob_ds_20 = xr.open_dataset('/Odyssey/private/ochapron/ConcreteAE/GS_model/outputs_v3/obs_prob_natl60_sigma_20.nc')

loss_ds_30 = xr.open_dataset('/Odyssey/private/ochapron/ConcreteAE/GS_model/outputs_v3/results_natl60_sigma_30.nc')
logits_ds_30 = xr.open_dataset('/Odyssey/private/ochapron/ConcreteAE/GS_model/outputs_v3/logits_natl60_sigma_30.nc')
prob_ds_30 = xr.open_dataset('/Odyssey/private/ochapron/ConcreteAE/GS_model/outputs_v3/obs_prob_natl60_sigma_30.nc')

loss_ds_40 = xr.open_dataset('/Odyssey/private/ochapron/ConcreteAE/GS_model/outputs_v3/results_natl60_sigma_40.nc')
logits_ds_40 = xr.open_dataset('/Odyssey/private/ochapron/ConcreteAE/GS_model/outputs_v3/logits_natl60_sigma_40.nc')
prob_ds_40 = xr.open_dataset('/Odyssey/private/ochapron/ConcreteAE/GS_model/outputs_v3/obs_prob_natl60_sigma_40.nc')

In [6]:
# Replace the time dimension and coordinates
loss_ds = loss_ds.assign_coords(time=tgt_ds.time)
logits_ds = logits_ds.assign_coords(time=tgt_ds.time)
prob_ds = prob_ds.assign_coords(time=tgt_ds.time)

time_5days = tgt_ds.time[::5]

loss_ds_2 = loss_ds_2.assign_coords(time=time_5days)
logits_ds_2 = logits_ds_2.assign_coords(time=time_5days)
prob_ds_2 = prob_ds_2.assign_coords(time=time_5days)

loss_ds_10 = loss_ds_10.assign_coords(time=time_5days)
logits_ds_10 = logits_ds_10.assign_coords(time=time_5days)
prob_ds_10 = prob_ds_10.assign_coords(time=time_5days)

loss_ds_15 = loss_ds_15.assign_coords(time=time_5days)
logits_ds_15 = logits_ds_15.assign_coords(time=time_5days)
prob_ds_15 = prob_ds_15.assign_coords(time=time_5days)

loss_ds_5 = loss_ds_5.assign_coords(time=time_5days)
logits_ds_5 = logits_ds_5.assign_coords(time=time_5days)
prob_ds_5 = prob_ds_5.assign_coords(time=time_5days)

loss_ds_20 = loss_ds_20.assign_coords(time=time_5days)
logits_ds_20 = logits_ds_20.assign_coords(time=time_5days)
prob_ds_20 = prob_ds_20.assign_coords(time=time_5days)

loss_ds_30 = loss_ds_30.assign_coords(time=time_5days)
logits_ds_30 = logits_ds_30.assign_coords(time=time_5days)
prob_ds_30 = prob_ds_30.assign_coords(time=time_5days)

loss_ds_40 = loss_ds_40.assign_coords(time=time_5days)
logits_ds_40 = logits_ds_40.assign_coords(time=time_5days)
prob_ds_40 = prob_ds_40.assign_coords(time=time_5days)

In [7]:
tgt_ds_on_logits = tgt_ds.sel(lat=logits_ds.lat, lon=logits_ds.lon,method="nearest")
tgt_ds_on_logits

<xarray.Dataset> Size: 688MB
Dimensions:       (lat: 280, lon: 280, time: 365)
Coordinates:
  * lat           (lat) float64 2kB 29.55 29.6 29.65 29.7 ... 43.4 43.45 43.5
  * lon           (lon) float64 2kB -64.45 -64.4 -64.35 ... -50.6 -50.55 -50.5
  * time          (time) datetime64[ns] 3kB 2012-10-01 2012-10-02 ... 2013-09-30
Data variables:
    mdt           (lat, lon) float64 627kB ...
    sla           (time, lat, lon) float64 229MB ...
    ssh           (time, lat, lon) float64 229MB ...
    ssh_norm      (time, lat, lon) float64 229MB ...
    ssh_variance  (lat, lon) float64 627kB ...
Attributes:
    About:    Created by SOSIE interpolation environement => https://github.c...
    Info:     Horizontal grid read in regulargrid_NATL60.nc / Source field re...

In [ ]:
# -*- coding: utf-8 -*-
"""
Optimal‑sensor mask analysis – **information‑theoretic perspective**
===================================================================
This refactor **removes** all gradient / Laplacian / raw‑moment metrics and
replaces them with Shannon–entropy and mutual‑information diagnostics that
quantify how much variability the mask captures.

Metrics collected per iteration
-------------------------------
* **H_obs**  – Shannon entropy of the SSH values *actually observed* by the mask.
* **H_diff** = H_gt – H_obs.
* **MI_mask** = H_gt – H_unobs   (mutual information between the mask and the full field).
* **LocH_obs**  – mean of the **local (window‑9) entropy map** at the observed pixels.
* **LocH_diff** = LocH_gt – LocH_obs.

We keep the same train/test loop structure:
  • 100 iterations with a Bernoulli‑random mask  → learned_flag = 0
  • 100 iterations with the trained mask         → learned_flag = 1

For each day we correlate those five metrics with RMSE & ExpVar and finally
produce two 5‑row × 1‑column scatter‑grid figures (one for each target metric).
"""

import numpy as np
import torch, torch.nn.functional as F
import scipy.ndimage as ndi
from scipy.stats import linregress
from skimage.util import view_as_windows
from collections import namedtuple
import matplotlib.pyplot as plt
import pandas as pd

# ─────────────────────────────────────────────────────────────────────────────
# Helper functions
# ─────────────────────────────────────────────────────────────────────────────
tau = 1
hard = True
padding_size = 10
rate = 0.999
def shannon_entropy(arr, bins=128, base=np.e):
    """Shannon entropy of a 1‑D array (ignores NaNs)."""
    a = np.asarray(arr).ravel()
    a = a[~np.isnan(a)]
    if a.size == 0:
        return np.nan
    hist, _ = np.histogram(a, bins=bins, density=True)
    p = hist[hist > 0]
    return -(p * np.log(p) / np.log(base)).sum()


def local_entropy_map(field, window=9, bins=64, base=np.e):
    """Return a 2‑D map of Shannon entropy in a sliding window."""
    pad = window // 2
    f   = np.pad(field, pad, mode='reflect')
    blocks = view_as_windows(f, (window, window))  # shape (lat, lon, w, w)
    H = np.empty(field.shape)
    for i in range(H.shape[0]):
        for j in range(H.shape[1]):
            H[i, j] = shannon_entropy(blocks[i, j], bins=bins, base=base)
    return H

# ─────────────────────────────────────────────────────────────────────────────
# Containers across all days / iterations
# ─────────────────────────────────────────────────────────────────────────────
losses_all, expvar_all, learned_flag_all = [], [], []
H_obs_all, H_diff_all, MI_all = [], [], []
LocH_obs_all, LocH_diff_all   = [], []

per_day_stats_RMSE, per_day_stats_Exp = [], []

# ─────────────────────────────────────────────────────────────────────────────
# PER‑DAY LOOP
# ─────────────────────────────────────────────────────────────────────────────
for day in time_5days:  
    TrainingItem = namedtuple('TrainingItem', ['input', 'tgt'])

    inp_da  = tgt_ds.ssh.sel(time=day)
    inp_da_GS = inp_da.sel(lat=slice(28, 45.), lon=slice(-66, -49.))

    border_size = 30
    lat_shape, lon_shape = len(inp_da_GS.lat), len(inp_da_GS.lon)
    inp_da_GS_crop = inp_da_GS.isel(lat=slice(border_size, lat_shape-border_size),
                                    lon=slice(border_size, lon_shape-border_size))

    inp_da_GS = inp_da_GS.fillna(0.0)
    time  = 1
    lat, lon = inp_da_GS_crop.shape

    mean_tgt, std_tgt = inp_da_GS_crop.mean().item(), inp_da_GS_crop.std().item()
    tens_inp_da = torch.from_numpy(inp_da_GS_crop.values).float().to(device)
    tens_inp_da = (tens_inp_da - mean_tgt) / std_tgt
    tens_inp_da = tens_inp_da.unsqueeze(0).unsqueeze(0)
    tens_inp_da.requires_grad_(True)

    tgt_inp      = tens_inp_da.clone()
    grid_size    = lat
    batch        = TrainingItem(input=tens_inp_da, tgt=tens_inp_da)

    # 2) Ground‑truth entropy maps -----------------------------------------
    ssh_np = tgt_tensor.cpu().numpy()
    H_gt       = shannon_entropy(ssh_np)
    LocH_gt_map = local_entropy_map(ssh_np, window=9, bins=64)
    LocH_gt     = np.nanmean(LocH_gt_map)

    # 3) Logits for random vs trained masks -------------------------------
    grid = lat
    logits_rand = torch.zeros((2,1,grid,grid), device=device)
    logits_rand[1] = np.log(rate / (1-rate))
    logits_trained = torch.zeros_like(logits_rand)
    logits_trained[1] = torch.from_numpy(logits_ds.logits.sel(time=day).values).float()

    length_scale = loss_ds.length_scale.sel(time=day).values

    # 4) Per‑day container
    per_day = {k: [] for k in ['H_obs','H_diff','MI','LocH_obs','LocH_diff','RMSE','ExpVar']}

    # 5) Iteration helper ---------------------------------------------------
    def iterate(logits, flag, ls):
        for _ in range(100):
            mask = F.gumbel_softmax(logits, hard=True, dim=0)[0].view(1,1,lat,lon)
            mask_np = mask[0,0].cpu().numpy() > 0.5

            # Optimal interpolation reconstruction
            sel = batch.tgt * mask
            sel[sel == 0] = float('nan')
            rec, _ = optimal_interpolation(sel[0,0], length_scale=ls)
            rec = rec.cpu().numpy()*std_tgt + mean_tgt
            gt   = batch.tgt.cpu().detach().numpy()*std_tgt + mean_tgt

            rmse = np.sqrt(np.nanmean((gt - rec)**2))
            expv = 1 - np.nanvar(rec - gt, ddof=1) / np.nanvar(gt, ddof=1)

            obs_vals = np.where(mask_np, ssh_np, np.nan)
            unobs_vals = np.where(~mask_np, ssh_np, np.nan)

            H_obs   = shannon_entropy(obs_vals)
            H_unobs = shannon_entropy(unobs_vals)
            MI      = H_gt - H_unobs
            H_diff  = H_gt - H_obs

            LocH_obs = np.nanmean(np.where(mask_np, LocH_gt_map, np.nan))
            LocH_diff = LocH_gt - LocH_obs

            # ‑‑ store global
            losses_all.append(rmse); expvar_all.append(expv); learned_flag_all.append(flag)
            H_obs_all.append(H_obs); H_diff_all.append(H_diff); MI_all.append(MI)
            LocH_obs_all.append(LocH_obs); LocH_diff_all.append(LocH_diff)

            # ‑‑ store per‑day
            per_day['RMSE'].append(rmse); per_day['ExpVar'].append(expv)
            per_day['H_obs'].append(H_obs); per_day['H_diff'].append(H_diff); per_day['MI'].append(MI)
            per_day['LocH_obs'].append(LocH_obs); per_day['LocH_diff'].append(LocH_diff)

    iterate(logits_rand,    flag=0, ls=length_scale[0])
    iterate(logits_trained, flag=1, ls=length_scale[-1])

    # 6) Per‑day correlations ----------------------------------------------
    def _corr(x,y):
        x,y = np.asarray(x), np.asarray(y)
        ok = ~(np.isnan(x)|np.isnan(y))
        return linregress(x[ok], y[ok]) if ok.sum()>1 else (np.nan,)*5

    for m in ['H_obs','H_diff','MI','LocH_obs','LocH_diff']:
        slope,r, *_ = _corr(per_day[m], per_day['RMSE'])[:3]
        per_day_stats_RMSE.append({'Day':day.values,'Metric':m,'Slope':slope,'R2':r**2})
        slope,r, *_ = _corr(per_day[m], per_day['ExpVar'])[:3]
        per_day_stats_Exp.append({'Day':day.values,'Metric':m,'Slope':slope,'R2':r**2})

# ─────────────────────────────────────────────────────────────────────────────
# 7) Global scatter‑plots – two separate figs (RMSE, ExpVar)
# ─────────────────────────────────────────────────────────────────────────────
metrics = [
    ("Entropy obs (H_obs)",   H_obs_all),
    ("Entropy diff (H_gt−obs)", H_diff_all),
    ("Mutual info (H_gt−H_unobs)", MI_all),
    ("Local‑H obs", LocH_obs_all),
    ("Local‑H diff", LocH_diff_all),
]
rows = len(metrics)

def make_fig(y, ylab):
    fig, axes = plt.subplots(rows, 1, figsize=(5, rows*2.2), sharex=False)
    learned = np.asarray(learned_flag_all)
    for i, (lab, xvals) in enumerate(metrics):
        ax = axes[i]
        x = np.asarray(xvals)
        yy = np.asarray(y)
        ok = ~(np.isnan(x)|np.isnan(yy))
        ax.scatter(x[ok & (learned==0)], yy[ok & (learned==0)], s=8, alpha=0.35, label='Random' if i==0 else None)
        ax.scatter(x[ok & (learned==1)], yy[ok & (learned==1)], s=8, alpha=0.35, color='red', label='Learned' if i==0 else None)
        slope, inter, r, *_ = linregress(x[ok], yy[ok])
        xs = np.linspace(np.nanmin(x[ok]), np.nanmax(x[ok]), 150)
        ax.plot(xs, slope*xs+inter, 'k-', lw=1)
        ax.text(0.03,0.95,f"m={slope:+.2e}\nR²={r**2:.2f}", transform=ax.transAxes, va='top', ha='left', fontsize=7,
                bbox=dict(boxstyle='round',fc='white',alpha=0.8,lw=0.4))
        ax.set_ylabel(ylab if i==0 else '')
        ax.set_title(lab, fontsize=9, pad=4)
        ax.grid(True, linewidth=0.3)
    axes[-1].set_xlabel('Metric value')
    fig.legend(loc='upper right', fontsize=8)
    fig.tight_layout(rect=[0,0,0.96,0.97])
    plt.show()

make_fig(losses_all, 'RMSE')
make_fig(expvar_all, 'Expl. Var')